# 09 — Spectral transformer on the mock cube
## Part A · From a spectrum to a stress logit

Build and check a small PyTorch encoder before the second day's training. Each token represents one measured wavelength of a centre pixel. Notebook 04's fixed band selection preserves physical wavelengths and their gaps. Notebook 06 supplies the spatial split convention. Run notebooks 03 and 04 first.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import torch
from torch import nn

root = Path(".") if Path("notebooks").is_dir() else Path("..")
with np.load(root / "data/processed/mock_pca.npz") as archive:
    valid = archive["valid"]
    kept_nm = archive["wavelengths_kept"]
with h5py.File(root / "data/mock/mock_cube.h5") as scene:
    wavelengths = scene["wavelengths_nm"][...]
    keep = np.isin(wavelengths, kept_nm)
    assert np.array_equal(wavelengths[keep], kept_nm)
    spectra = scene["cube"][...][..., keep]
    labels = scene["labels"][...]
assert spectra.shape[:2] == valid.shape == labels.shape
split_row = 38
train_spectra = spectra[:split_row][valid[:split_row]]
train_labels = labels[:split_row][valid[:split_row]]
mean = train_spectra.mean(axis=0, dtype=np.float64)
scale = train_spectra.std(axis=0, dtype=np.float64).clip(min=1e-8)
train_x = torch.from_numpy(((train_spectra - mean) / scale).astype(np.float32))
train_y = torch.from_numpy(train_labels.astype(np.float32))
print(f"{len(train_x)} training centres, {len(kept_nm)} wavelengths, {int(train_y.sum())} stressed centres")

2546 training centres, 166 wavelengths, 8 stressed centres


### Embed each wavelength

A token carries two values: standardised reflectance and its wavelength rescaled to the interval [−1, 1]. A linear projection embeds the pair in 32 dimensions. Wavelength identifies a token even across the water-absorption gaps removed in notebook 04. Four attention heads let tokens exchange information across the spectrum. Mean pooling then gives one vector per centre pixel.

In [2]:
class SpectralTransformer(nn.Module):
    def __init__(self, wavelengths_nm, width=32, heads=4):
        super().__init__()
        wave = torch.as_tensor(wavelengths_nm, dtype=torch.float32)
        assert wave.ndim == 1 and torch.all(torch.diff(wave) > 0)
        positions = 2 * (wave - wave.min()) / (wave.max() - wave.min()) - 1
        self.register_buffer("positions", positions)
        self.embedding = nn.Linear(2, width)
        block = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=2 * width,
            dropout=0.1, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(block, num_layers=1, enable_nested_tensor=False)
        self.classifier = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 1))

    def tokens(self, spectrum):
        if spectrum.ndim != 2 or spectrum.shape[1] != len(self.positions):
            raise ValueError("Expected one reflectance value per retained wavelength.")
        pairs = torch.stack((spectrum, self.positions.expand_as(spectrum)), dim=-1)
        return self.embedding(pairs)

    def forward(self, spectrum):
        features = self.encoder(self.tokens(spectrum))
        return self.classifier(features.mean(dim=1)).squeeze(-1)


torch.manual_seed(0)
model = SpectralTransformer(kept_nm)
indices = torch.cat([torch.where(train_y == k)[0][:4] for k in (0, 1)])
xb, yb = train_x[indices], train_y[indices]
assert torch.isfinite(xb).all() and xb.shape == (8, len(kept_nm))
print("Spectra", tuple(xb.shape), "| tokens", tuple(model.tokens(xb).shape), "| logits", tuple(model(xb).shape))
print("Trainable parameters", sum(p.numel() for p in model.parameters() if p.requires_grad))

Spectra (8, 166) | tokens (8, 166, 32) | logits (8,)
Trainable parameters 8737


### Check attention and gradients

In evaluation mode, inspect the attention weights for one spectrum. Each head assigns a distribution over input wavelengths for every query wavelength. Then run a single backward pass on eight training examples to verify that the loss reaches the embedding, encoder and classifier. This is a wiring check.

In [3]:
model.eval()
block = model.encoder.layers[0]
with torch.no_grad():
    token = model.tokens(xb[:1])
    _, weights = block.self_attn(token, token, token, need_weights=True, average_attn_weights=False)
    assert weights.shape == (1, 4, len(kept_nm), len(kept_nm))
    torch.testing.assert_close(weights.sum(-1), torch.ones_like(weights.sum(-1)))
    torch.testing.assert_close(model(xb)[:1], model(xb[:1]), atol=1e-5, rtol=1e-5)
print("Attention (batch, heads, query bands, key bands)", tuple(weights.shape))

model.train()
model.zero_grad(set_to_none=True)
logits = model(xb)
assert logits.shape == yb.shape
loss = nn.BCEWithLogitsLoss()(logits, yb)
loss.backward()
for name, param in model.named_parameters():
    assert param.grad is not None and torch.isfinite(param.grad).all() and param.grad.norm() > 0, name
print(f"Finite loss {loss.item():.4f}; every trainable parameter received a gradient.")

Attention (batch, heads, query bands, key bands) (1, 4, 166, 166)
Finite loss 0.7964; every trainable parameter received a gradient.


### Handoff to Part B

The encoder accepts one spectrum per centre and returns one binary logit. Part B can initialise a fresh model, train on the valid rows above 38 with class weighting, then score the held-out rows using the fixed training band statistics. Use the same validation centres and metrics as notebook 08 for the comparison.